# Node Type GNN — train and cross-validate

Predicts a per-node type (background / epithelial / hyphal) from the node embeddings
alongside the existing edge classifier, with a combined loss, so the model learns a joint
representation. Nothing is fed in explicitly: that a true edge cannot span background↔cell
or epithelial↔hyphal is left for the network to learn.

Labels were chosen in `11_Node Type Classification.ipynb`, which is the record of how and
why. This notebook consumes those decisions and trains.

- background: a fragment overlapping its best-matching GT cell by less than
  `BACKGROUND_THRESH` of its own area.
- epithelial / hyphal: the type of the GT cell a fragment was assigned to, from that cell's
  `mean_width`, per image because magnification differs.

The same `BACKGROUND_THRESH` produces the node types AND the edge labels — one split feeds
both.

`K = 10`, matching `10_Merge Oversegmentation GNN.ipynb`: k=7 left 22 of 483 true merges
(4.6%) outside the candidate set where no model can predict them; k=10 leaves 6 (1.2%).

That notebook is this one's baseline. It runs the same `k`, the same
`min_overlap_frac`, and the same visual branch, with `predict_node_type=False` — so a
difference between the two is attributable to the node head. Its older `merge_cv` run
used k=7 / 0.5 and is NOT comparable to either.


In [ ]:
%load_ext autoreload
%autoreload 2


## 1. Load images, AIS predictions, GT masks and cell-type rules


In [ ]:
import json
from pathlib import Path

import numpy as np
import tifffile

from image_processing_tools.image_class.image_container import ImageContainer

DATA_CONFIG = "merge_oversegmentation_data.json"
MICROSAM_ENV = "microsam"   # conda env that can read the v3 embedding zarr and stitch it
DATASET_ROOT = ("/home/wangchuyao/Projects/C_Albicans Thesis Project/7. Data/"
                "graph_datasets/node_type")

BACKGROUND_THRESH = 0.1    # one split feeds node types AND edge labels
K = 10                     # kNN candidates per fragment; matches notebook 10

with open(DATA_CONFIG) as f:
    SAMPLES = json.load(f)["samples"]


def _config(correct_dic_shift):
    """Per-sample preprocessing. Mirrors notebooks 10 and 11 so the fragments match."""
    return {
        "preprocessing": {
            "resize_image": False,
            "max_dim": 1080,
            "outlier_percentile": 0.35,
            "quantization": "16bit",
            "correct_DIC_shift": correct_dic_shift,
        }
    }


def load_label_map(path):
    lab = np.squeeze(tifffile.imread(str(Path(path).expanduser())))
    assert lab.ndim == 2, f"Expected a 2D label map, got shape {lab.shape} for {path}"
    return lab.astype(np.int32)


ais_labels, gt_labels, intensity_images, display_images = [], [], [], []
for s in SAMPLES:
    channels = [Path(p).expanduser() for p in s["image"].values()]
    config = _config(s.get("correct_DIC_shift", [0, 0]))
    # Grouped -> _sum_channels reduces to one 2D channel, summed and stretched. This is
    # what the intensity features are read from.
    intensity_images.append(ImageContainer([channels], config).merge())
    # Ungrouped -> the channels stay separate as (H, W, C). This becomes data.image, the
    # backdrop for the logged prediction overlays and the 2x2 merge figure. Omit it and
    # _log_figures skips both SILENTLY -- has_pred_figs tests hasattr(data, 'image').
    display_images.append(ImageContainer(channels, config).merge())
    ais_labels.append(load_label_map(s["ais"]))
    gt_labels.append(load_label_map(s["label"]))

missing = [i for i, s in enumerate(SAMPLES) if "cell_type" not in s]
if missing:
    raise ValueError(f"Samples {missing} have no `cell_type` rule; add one in {DATA_CONFIG}.")

print(f"Loaded {len(ais_labels)} images.")
for i, (a, g, im, disp) in enumerate(zip(ais_labels, gt_labels, intensity_images, display_images)):
    assert a.shape == g.shape and im.shape == a.shape, f"img {i}: shape mismatch"
    print(f"  img {i}: AIS fragments={len(np.unique(a))-1:>4}  GT cells={len(np.unique(g))-1:>3}"
          f"  display={disp.shape}  rule={SAMPLES[i]['cell_type']}")


## 2. Build the graphs, with node types attached

`build_cell_graph_data` derives both label sets from the same `min_overlap_frac`: the node
types via the assigned GT cell's shape, and the edge labels via the per-GT-cell MST over the
non-background fragments.

Images 0 and 1 must show **zero epithelial nodes** — they contain no epithelial cells. That
is the data, not a bug, and it is why per-class metrics are reported only for the classes
present in a fold.


In [ ]:
from image_processing_tools.scene_graph_network.build_cell_dataset import build_cell_graph_data
from image_processing_tools.scene_graph_network.gnn_data import save_pyg_dataset
from image_processing_tools.scene_graph_network.precompute_microsam_feats import load_or_stitch_embeddings

emb_dir = Path(DATASET_ROOT) / "embeddings"


def _embedding_path(sample):
    """Absolute path to a sample's tiled .zarr store, or None.

    Paths in the JSON are `~`-relative; zarr does no tilde expansion of its own and would
    look for a directory literally named "~".
    """
    emb = sample.get("embedding")
    return Path(emb).expanduser() if emb else None


def _npz_name(embedding_path):
    """Cache name for a stitched embedding, keyed on the .zarr's own name -- never on the
    sample's position in SAMPLES, since reordering the JSON would then hand a cached map to
    a different image with no error."""
    return Path(embedding_path).stem + "_microsam_features.npz"


_names = [_npz_name(p) for p in (_embedding_path(s) for s in SAMPLES) if p]
assert len(_names) == len(set(_names)), (
    f"Embedding stores map to duplicate cache names: "
    f"{[n for n in _names if _names.count(n) > 1]}"
)

data_list = []
for i, s in enumerate(SAMPLES):
    emb = _embedding_path(s)
    if emb is None:
        raise ValueError(f"sample {i} has no `embedding`; the node head uses the visual "
                         f"branch, so every sample needs one.")
    if not emb.exists():
        raise FileNotFoundError(f"sample {i}: embedding store not found: {emb}")
    # Stitches the tiled .zarr in-process, or via `conda run -n MICROSAM_ENV` when this
    # env's zarr cannot read the store. Cached: stitches at most once per store.
    npz_path = load_or_stitch_embeddings(str(emb), save_path=str(emb_dir),
                                         npz_name=_npz_name(emb), env=MICROSAM_ENV)
    data = build_cell_graph_data(
        ais_labels[i], intensity_images[i], gt_labels=gt_labels[i], k=K,
        min_overlap_frac=BACKGROUND_THRESH,     # feeds node types AND edge labels
        cell_type_rule=s["cell_type"],
        microsam_npz_path=npz_path,             # the visual branch
        display_image=display_images[i],        # -> data.image. Without it the prediction
                                                # overlays and the 2x2 merge figure are
                                                # silently skipped, with no warning.
    )
    data_list.append(data)

print(f"{'img':<5}{'nodes':>7}{'bg':>6}{'epi':>6}{'hyph':>6}{'edges':>8}{'positive':>10}")
print("-" * 48)
for i, d in enumerate(data_list):
    c = np.bincount(d.node_type.numpy(), minlength=3)
    print(f"{i:<5}{d.num_nodes:>7}{c[0]:>6}{c[1]:>6}{c[2]:>6}"
          f"{d.edge_index.shape[1]:>8}{int(d.edge_label.sum()):>10}")

save_pyg_dataset(data_list, DATASET_ROOT)
print(f"\nSaved {len(data_list)} graphs to {DATASET_ROOT}/processed/data.pt")


## 3. Overfit test

Can the model fit both label sets on a single graph? Image 3 is used because it carries all
three node classes in quantity. A high edge AUC together with a high node accuracy confirms
the labels and both heads are learnable before spending time on cross-validation.

`node_loss_weight` trades off how much the shared representation serves each task. Both
losses are per-sample means, so it is not compensating for the differing sample counts.


In [ ]:
from image_processing_tools.scene_graph_network.gnn_train import train_overfit_test

# Identical to 10_Merge Oversegmentation GNN.ipynb except for the node head, so a
# difference between the two runs is attributable to it and nothing else.
model_params = {
    "hidden_channels": 64,
    "dropout_p": 0.0,
    "use_visual_features": True,    # the node head is meant to use the visual branch
    "node_feature_dim": 8,
    "edge_feature_dim": 10,
    "predict_node_type": True,
    "num_node_classes": 3,
    # visual-branch params (must match notebook 10's):
    "d_visual": 32,
    "node_bbox_pad_frac": 0.1,
    "edge_box_margin_frac": 0.15,
    "edge_box_margin_floor": 20,
    "roi_output_size": 7,
}

results = train_overfit_test(
    dataset=[data_list[3]],         # all three node classes well represented
    max_epochs=1000,
    batch_size=1,
    learning_rate=0.01,
    model_params=model_params,
    patience=500,
    neg_sample_ratio=1.0,
    node_loss_weight=1.0,
    node_sample_ratio=1.0,          # equal counts per present class
    experiment="nodetype_overfit_one_graph_k10_minFrac0_1_visual",
)
print(results)


## 4. Cross-validation

Leave-one-out: each fold trains on five images and is scored on the held-out one, so the
reported numbers are generalisation to an unseen image rather than fit.

Node metrics are reported per fold only for the classes present in that fold's labels, and
the across-fold aggregate averages each class only over the folds where it appeared, naming
the count. Folds holding out image 0 or 1 cannot score epithelial at all.


In [ ]:
from image_processing_tools.scene_graph_network.gnn_train import n_fold_validation

NUM_FOLDS = len(data_list)          # leave-one-out
assert NUM_FOLDS >= 2, f"Cross-validation needs >= 2 graphs, dataset has {NUM_FOLDS}."

cv_results = n_fold_validation(
    dataset=data_list,
    num_folds=NUM_FOLDS,
    max_epochs=1000,
    batch_size=1,
    learning_rate=0.01,
    model_params=model_params,      # same architecture as the overfit test
    patience=200,
    min_epoch=50,                   # suppress early stopping until a high AUC is not luck
    neg_sample_ratio=1.0,
    label_smoothing=0.1,
    node_loss_weight=1.0,
    node_sample_ratio=1.0,
    experiment="nodetype_cv_k10_minFrac0_1_visual",   # new subfolder; the
                                # earlier nodetype_cv ran without the visual
                                # branch and before the edge_attr direction fix
)

for i, r in enumerate(cv_results, start=1):
    node = f" node_acc={r['node_accuracy']:.4f}" if "node_accuracy" in r else ""
    print(f"fold {i}: AUC={r['test_auc']:.4f} acc={r['test_acc']:.4f} "
          f"thresh={r['best_threshold']:.4f}{node}")
